# Image-only OCR: TyphoonOCR-2B + Qwen-VL 3.0

Notebook นี้อ่านเฉพาะไฟล์รูปภาพจาก `renders/` แล้วใช้ `typhoonOCR-2B` เพื่อ OCR ก่อนส่งผลลัพธ์ให้ `Qwen-VL 3.0` ช่วยแก้คำผิด จัดระเบียบ แยกข้อมูลสำคัญ และสรุปผล

ผลลัพธ์ที่สร้าง:
- `ocr_results.json`
- `ocr_results.csv`
- `ocr_report.md`

หมายเหตุ: โมเดล vision-language ขนาด 2B/8B ใช้ VRAM/RAM สูงและต้องดาวน์โหลดจาก Hugging Face ในครั้งแรก

## Setup

รัน cell ด้านล่างถ้ายังไม่ได้ติดตั้ง dependencies ใน environment ปัจจุบัน

In [ ]:
# Optional dependency install. Uncomment and run once if your environment is missing packages.
# %pip install -U "transformers>=4.57.0" accelerate pillow pandas tqdm torch torchvision sentencepiece protobuf qwen-vl-utils

In [ ]:
from __future__ import annotations

import gc
import json
import re
import traceback
from pathlib import Path
from typing import Any
print("Imports successful.")

import pandas as pd
import torch
print("More imports successful.")
from PIL import Image, UnidentifiedImageError
print("More imports successful.")
from tqdm.auto import tqdm
print("More imports successful.")
from transformers import AutoModelForCausalLM, AutoModelForVision2Seq, AutoProcessor
print("All imports successful.")

# Paths
PROJECT_ROOT = Path.cwd()
RENDERS_DIR = PROJECT_ROOT / "renders"
print("Paths set.")

# Only these extensions are processed. Everything else under renders/ is ignored.
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}

# Model IDs. You can switch Qwen to a larger model if your GPU can handle it.
TYPHOON_MODEL_ID = "typhoon-ai/typhoon-ocr1.5-2b"
QWEN_MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"

# Processing controls
MAX_IMAGES = None  # set to a small number such as 5 for a quick test
USE_QWEN_IMAGE_CONTEXT = True  # send both OCR text and image to Qwen-VL when possible

# Generation controls
TYPHOON_MAX_NEW_TOKENS = 10000
QWEN_MAX_NEW_TOKENS = 4096

# Output files
OUTPUT_JSON = PROJECT_ROOT / "ocr_results.json"
OUTPUT_CSV = PROJECT_ROOT / "ocr_results.csv"
OUTPUT_MD = PROJECT_ROOT / "ocr_report.md"
print("Configuration complete.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Renders dir : {RENDERS_DIR}")
print(f"Device      : {'cuda' if torch.cuda.is_available() else 'cpu'}")

Imports successful.
More imports successful.
More imports successful.
More imports successful.


## Load Images

สแกน `renders/` แบบ recursive แต่รับเฉพาะ image extensions ที่กำหนดไว้เท่านั้น

In [ ]:
if not RENDERS_DIR.exists():
    raise FileNotFoundError(
        f"ไม่พบโฟลเดอร์ renders/: {RENDERS_DIR}\n"
        "กรุณารัน notebook จาก root ของ data bundle หรือแก้ค่า RENDERS_DIR ให้ถูกต้อง"
    )

all_files = sorted(path for path in RENDERS_DIR.rglob("*") if path.is_file())
image_paths = [path for path in all_files if path.suffix.lower() in IMAGE_EXTENSIONS]
ignored_files = [path for path in all_files if path.suffix.lower() not in IMAGE_EXTENSIONS]

if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

if not image_paths:
    raise RuntimeError(
        "ไม่พบไฟล์รูปภาพใน renders/\n"
        "รองรับเฉพาะ .png, .jpg, .jpeg, .webp และ notebook นี้จะไม่อ่าน PDF, DOCX, TXT หรือไฟล์อื่น"
    )

print(f"Found image files : {len(image_paths):,}")
print(f"Ignored non-images: {len(ignored_files):,}")
print("Sample images:")
for path in image_paths[:5]:
    print("-", path.relative_to(PROJECT_ROOT))

## OCR With TyphoonOCR-2B

โหลดโมเดล OCR และอ่านข้อความจากรูปทีละไฟล์ หากไฟล์ใด fail จะเก็บ error ไว้ใน record แล้วไปไฟล์ถัดไป

In [ ]:
TYPHOON_OCR_PROMPT = """Extract all text from the image.

Instructions:
- Only return the clean Markdown.
- Do not include any explanation or extra text.
- You must include all information on the page.

Formatting Rules:
- Tables: Render tables using <table>...</table> in clean HTML format.
- Equations: Render equations using LaTeX syntax with inline ($...$) and block ($$...$$).
- Images/Charts/Diagrams: Wrap any clearly defined visual areas in <figure>...</figure> and describe them in Thai.
- Page Numbers: Wrap page numbers in <page_number>...</page_number>.
- Checkboxes: Use ☐ for unchecked and ☑ for checked boxes.
"""

QWEN_CLEANUP_PROMPT_TEMPLATE = """You are a careful Thai/English document OCR reviewer.

Use the image and/or OCR text to correct OCR mistakes, organize the content, identify fields/tables/topics, and summarize key information.
Do not invent facts that are not visible in the image or present in the OCR text.

Return JSON only with this schema:
{{
  "cleaned_text": "corrected readable text in Markdown",
  "structured_result": {{
    "document_type": "best guess or unknown",
    "topics": [],
    "important_fields": {{}},
    "tables": []
  }},
  "summary": "concise Thai summary",
  "corrections": ["important OCR corrections or uncertainties"],
  "confidence_notes": "short note about confidence/legibility"
}}

File: {file_name}

Raw OCR text:
```markdown
{raw_ocr_text}
```
"""

def load_image(path: Path) -> Image.Image:
    """Load image as RGB and fail cleanly for corrupt/non-image files."""
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError) as exc:
        raise RuntimeError(f"เปิดรูปภาพไม่ได้: {path}") from exc


def resize_for_typhoon(img: Image.Image, target_long_side: int = 1800, threshold: int = 300) -> Image.Image:
    """Typhoon OCR v1.5 is trained around a 1800px image dimension."""
    width, height = img.size
    long_side = max(width, height)
    if long_side <= threshold:
        return img
    scale = target_long_side / float(long_side)
    new_size = (max(1, int(width * scale)), max(1, int(height * scale)))
    return img.resize(new_size, Image.Resampling.LANCZOS)


def load_image_text_model(model_id: str):
    """Load a Hugging Face VLM/text model without importing AutoModelForImageTextToText.

    Some torch/transformers combinations fail while importing AutoModelForImageTextToText
    with: cannot import name 'setup_compilation_env' from torch._higher_order_ops.utils.
    Using AutoModelForVision2Seq first, then AutoModelForCausalLM, avoids that brittle import.
    """
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    errors = []

    for model_cls in (AutoModelForVision2Seq, AutoModelForCausalLM):
        try:
            model = model_cls.from_pretrained(
                model_id,
                dtype="auto",
                device_map="auto",
                trust_remote_code=True,
            )
            model.eval()
            return processor, model
        except TypeError:
            try:
                model = model_cls.from_pretrained(
                    model_id,
                    torch_dtype="auto",
                    device_map="auto",
                    trust_remote_code=True,
                )
                model.eval()
                return processor, model
            except Exception as exc:
                errors.append(f"{model_cls.__name__}: {exc}")
        except Exception as exc:
            errors.append(f"{model_cls.__name__}: {exc}")

    raise RuntimeError(
        "โหลดโมเดลไม่สำเร็จ อาจเป็นเพราะ torch/transformers ไม่เข้ากัน หรือ model id ต้องใช้ transformers เวอร์ชันใหม่กว่า.\n"
        + "\n".join(errors[-4:])
    )


def move_inputs_to_model_device(inputs: Any, model: Any):
    """Move BatchFeature inputs to a sensible device for non-sharded local inference."""
    model_device = getattr(model, "device", None)
    if model_device is None or str(model_device) == "meta":
        return inputs
    try:
        return inputs.to(model_device)
    except Exception:
        return inputs


def generate_vlm_text(processor: Any, model: Any, image: Image.Image | None, prompt: str, max_new_tokens: int) -> str:
    """Run chat-template generation for image+text or text-only prompts."""
    content = []
    if image is not None:
        content.append({"type": "image", "image": image})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = move_inputs_to_model_device(inputs, model)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[-1]
    generated_ids_trimmed = [out_ids[input_len:] for out_ids in generated_ids]
    text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]
    return text.strip()


def extract_json_object(text: str) -> dict[str, Any]:
    """Parse JSON output from Qwen, tolerating code fences or surrounding text."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except json.JSONDecodeError:
                pass
    return {
        "cleaned_text": cleaned,
        "structured_result": {},
        "summary": "",
        "corrections": [],
        "confidence_notes": "Qwen output was not valid JSON; stored raw text in cleaned_text.",
    }


def free_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
print(f"Loading Typhoon OCR model: {TYPHOON_MODEL_ID}")
typhoon_processor, typhoon_model = load_image_text_model(TYPHOON_MODEL_ID)
print("Typhoon OCR model loaded.")

In [ ]:
records: list[dict[str, Any]] = []

for image_path in tqdm(image_paths, desc="OCR with TyphoonOCR-2B"):
    rel_path = image_path.relative_to(PROJECT_ROOT).as_posix()
    print(f"OCR: {rel_path}")

    record = {
        "file_name": image_path.name,
        "file_path": rel_path,
        "raw_ocr_text": "",
        "cleaned_text": "",
        "structured_result": {},
        "summary": "",
        "status": "pending",
        "error": "",
        "image_width": None,
        "image_height": None,
    }

    try:
        image = load_image(image_path)
        record["image_width"], record["image_height"] = image.size
        ocr_image = resize_for_typhoon(image)
        record["raw_ocr_text"] = generate_vlm_text(
            typhoon_processor,
            typhoon_model,
            ocr_image,
            TYPHOON_OCR_PROMPT,
            max_new_tokens=TYPHOON_MAX_NEW_TOKENS,
        )
        record["status"] = "ocr_done"
    except Exception as exc:
        record["status"] = "ocr_failed"
        record["error"] = f"{type(exc).__name__}: {exc}"
        print(f"OCR failed for {rel_path}: {record['error']}")

    records.append(record)

print(f"OCR stage finished. Records: {len(records):,}")

In [ ]:
# Free memory before loading Qwen-VL.
del typhoon_model, typhoon_processor
free_gpu_memory()
print("Released Typhoon OCR model from memory.")

## Clean/Structure With Qwen-VL 3.0

ส่ง raw OCR text และรูปภาพให้ Qwen-VL 3.0 เพื่อแก้คำผิด จัดรูปแบบ แยก field/table และสรุปข้อมูล

In [ ]:
print(f"Loading Qwen-VL model: {QWEN_MODEL_ID}")
qwen_processor, qwen_model = load_image_text_model(QWEN_MODEL_ID)
print("Qwen-VL model loaded.")

In [ ]:
for record in tqdm(records, desc="Clean and structure with Qwen-VL"):
    image_path = PROJECT_ROOT / record["file_path"]
    print(f"Qwen-VL: {record['file_path']}")

    try:
        image = load_image(image_path) if USE_QWEN_IMAGE_CONTEXT else None
        prompt = QWEN_CLEANUP_PROMPT_TEMPLATE.format(
            file_name=record["file_name"],
            raw_ocr_text=record.get("raw_ocr_text", "") or "[OCR failed or empty]",
        )
        qwen_text = generate_vlm_text(
            qwen_processor,
            qwen_model,
            image,
            prompt,
            max_new_tokens=QWEN_MAX_NEW_TOKENS,
        )
        parsed = extract_json_object(qwen_text)
        record["cleaned_text"] = parsed.get("cleaned_text", "")
        record["structured_result"] = parsed.get("structured_result", {})
        record["summary"] = parsed.get("summary", "")
        record["qwen_corrections"] = parsed.get("corrections", [])
        record["qwen_confidence_notes"] = parsed.get("confidence_notes", "")
        record["qwen_raw_output"] = qwen_text
        record["status"] = "done" if record["status"] == "ocr_done" else "qwen_done_after_ocr_failed"
    except Exception as exc:
        previous_error = record.get("error", "")
        qwen_error = f"Qwen {type(exc).__name__}: {exc}"
        record["status"] = "qwen_failed" if record["status"] == "ocr_done" else "ocr_and_qwen_failed"
        record["error"] = "; ".join(part for part in [previous_error, qwen_error] if part)
        record["qwen_traceback"] = traceback.format_exc(limit=3)
        print(f"Qwen-VL failed for {record['file_path']}: {qwen_error}")

print("Qwen-VL stage finished.")

In [ ]:
# Free model memory after inference.
del qwen_model, qwen_processor
free_gpu_memory()
print("Released Qwen-VL model from memory.")

## Save Results

บันทึกผลลัพธ์เป็น JSON, CSV และ Markdown report

In [ ]:
# JSON keeps nested structured_result intact.
with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

# CSV stores nested objects as JSON strings for spreadsheet compatibility.
df = pd.DataFrame(records)
if "structured_result" in df.columns:
    df["structured_result"] = df["structured_result"].apply(lambda value: json.dumps(value, ensure_ascii=False))
if "qwen_corrections" in df.columns:
    df["qwen_corrections"] = df["qwen_corrections"].apply(lambda value: json.dumps(value, ensure_ascii=False))
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

# Markdown report for quick human review.
report_lines = ["# OCR Report", ""]
for item in records:
    report_lines.extend([
        f"## {item['file_path']}",
        f"- Status: {item.get('status', '')}",
        f"- Summary: {item.get('summary', '')}",
        "",
        "### Cleaned Text",
        item.get("cleaned_text") or "",
        "",
        "### Raw OCR Text",
        item.get("raw_ocr_text") or "",
        "",
    ])
    if item.get("error"):
        report_lines.extend(["### Error", item["error"], ""])

OUTPUT_MD.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Saved JSON    : {OUTPUT_JSON}")
print(f"Saved CSV     : {OUTPUT_CSV}")
print(f"Saved Markdown: {OUTPUT_MD}")

## Preview Results

ดูตัวอย่างผลลัพธ์และสถานะการประมวลผล

In [ ]:
preview_columns = [
    "file_name",
    "file_path",
    "status",
    "summary",
    "raw_ocr_text",
    "cleaned_text",
    "structured_result",
    "error",
]
available_columns = [column for column in preview_columns if column in df.columns]
display(df[available_columns].head(10))

print("Status counts:")
display(df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))